# This is an experimental notebook

In [1]:
# necessary imports
import pandas as pd 
from pathlib import Path
import wfdb
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
    BatchNormalization, Activation, AveragePooling2D,
    Dropout, Flatten, Dense, GlobalAveragePooling2D
)
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    cohen_kappa_score
    
)
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils import resample
from sklearn.model_selection import StratifiedKFold

c:\Users\ZEYNEP\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
import sys
from pathlib import Path

# Add src directory to path
src_path = Path("../").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Added to path: {src_path}")

Added to path: C:\Users\ZEYNEP\OneDrive\Desktop\CTG_Dissertation


In [3]:
# paths 
raw_dataset = Path("data/raw_dataset")

records = [p.stem for p in raw_dataset.glob("*.hea")]
print(f"Found {len(records)} CTG records")


Found 552 CTG records


In [4]:
# Read one record to test
record = wfdb.rdrecord(raw_dataset / records[0])

signals = record.p_signal        
signal_names = record.sig_name   
fs = record.fs                  
print(f"Signal names: {signal_names}, Sampling rate: {fs} Hz")

Signal names: ['FHR', 'UC'], Sampling rate: 4 Hz


In [5]:
# Convert signals to a list
fhr = signals[:, 0].tolist()
uc = signals[:, 1].tolist()
print(f"First 10 FHR values: {fhr[:10]}")
print(f"First 10 UC values: {uc[:10]}")

First 10 FHR values: [150.5, 150.5, 151.0, 151.25, 151.25, 150.25, 150.25, 150.25, 148.75, 148.75]
First 10 UC values: [7.0, 8.5, 8.5, 7.5, 9.5, 8.5, 10.5, 12.0, 11.0, 11.5]


In [6]:
'''
Step 2: Signal cleaning 
Repeated zero signals at the end of the samples were removed
'''
def remove_trailing_zeros(signal):
    """Remove trailing zeros from a signal."""
    if not isinstance(signal, list):
        raise ValueError("Input signal must be a list.")
    
    # Find the index of the last non-zero element
    last_non_zero_index = len(signal) - 1
    while last_non_zero_index >= 0 and signal[last_non_zero_index] == 0:
        last_non_zero_index -= 1
    
    # Return the signal up to the last non-zero element
    return signal[:last_non_zero_index + 1]

In [7]:
'''
Step 3: Signal Extraction
The 30 minutes immediately preceding the last non-zero signals were extracted.
'''
def extract_last_30_minutes(signal, sampling_rate=4):
    ''' reject short recordings and extract last 30 minutes of signal '''
    num_samples_30_minutes = 30 * 60 * sampling_rate

    if len(signal) < num_samples_30_minutes:
        return []  # reject short recordings

    return signal[-num_samples_30_minutes:]


In [8]:
'''
Step 4: Downsample signals to 1 Hz
The signals were originally sampled at 4 Hz. They were downsampled to 1 Hz by taking every fourth sample.
Paper explicitly states: “Signals were downsampled to 1 Hz for 30 minutes (1800 points)”
'''
def downsample_to_1hz(signal, original_fs=4, target_fs=1):
    factor = original_fs // target_fs
    return signal[::factor]


In [9]:
'''
Step 5: Selection
Only cases that satisfied the selection criteria (specifically, a signal loss less than 16%) were used for the final analysis
'''
def is_signal_acceptable(signal, threshold=0.16):
    """Check if the signal loss is within the acceptable threshold."""
    
    # Reject empty or invalid signals
    if not signal or len(signal) == 0:
        return False
    
    total_length = len(signal)
    zero_count = signal.count(0)
    signal_loss = zero_count / total_length
    
    return signal_loss < threshold


In [10]:
''' 
Step 6: Perform the above steps on the dataset 
'''
processed_records = []

for rec in records:
    record = wfdb.rdrecord(raw_dataset / rec)
    signals = record.p_signal

    fhr = signals[:, 0].tolist()
    uc  = signals[:, 1].tolist()

    # Cleaning 
    fhr = remove_trailing_zeros(fhr)
    uc  = remove_trailing_zeros(uc)

    # Extraction 
    fhr = extract_last_30_minutes(fhr, sampling_rate=record.fs)
    uc  = extract_last_30_minutes(uc, sampling_rate=record.fs)

    # Downsampling
    fhr = downsample_to_1hz(fhr, original_fs=fs, target_fs=1)
    uc  = downsample_to_1hz(uc, original_fs=fs, target_fs=1)

    # Selection
    if is_signal_acceptable(fhr) and is_signal_acceptable(uc):
        processed_records.append({
            "rec_id": rec,
            "FHR": fhr,
            "UC": uc
        })
print(f"Processed {len(processed_records)} records after cleaning and selection.")

Processed 220 records after cleaning and selection.


In [11]:
''' 
Classification: Map signals to step3_labels based on record IDs
'''
# path to majority voting labels
Labels = 'ExpertAnnotations\step3_labels.csv'

# print number of normal and abnormal deliveries
labels_df = pd.read_csv(Labels)
num_normal = sum(labels_df['Clinical_Label'] == 'No Hypoxia (Normal)')
num_suspicious = sum(labels_df['Clinical_Label'] == 'Mild Hypoxia (Suspicious)')
num_severe = sum(labels_df['Clinical_Label'] == 'Severe Hypoxia (Pathological)')
num_uninterpretable = sum(labels_df['Clinical_Label'] == 'Uninterpretable (Filtered)')
print(f'Number of normal deliveries: {num_normal}')
print(f'Number of suspicious deliveries: {num_suspicious}')
print(f'Number of severe deliveries: {num_severe}')
print(f'Number of uninterpretable deliveries: {num_uninterpretable}')

Number of normal deliveries: 127
Number of suspicious deliveries: 153
Number of severe deliveries: 57
Number of uninterpretable deliveries: 215


In [12]:
data_df = pd.DataFrame(processed_records)
print(data_df.shape)

(220, 3)


In [13]:
'''
Map the labels
'''

# Create a mapping from record ID to clinical label
labels_df = pd.read_csv(Labels)
print(labels_df.columns)

data_df['rec_id'] = data_df['rec_id'].astype(str)
labels_df['rec_id'] = labels_df['rec_id'].astype(str)
merged_df = data_df.merge(
    labels_df,
    left_on="rec_id",
    right_on="rec_id",
    how="inner"
)

print(merged_df.shape)
label_counts = merged_df['Clinical_Label'].value_counts()
print(label_counts)


Index(['rec_id', 'Majority_Vote_Label', 'Clinical_Label'], dtype='object')
(220, 5)
Clinical_Label
Uninterpretable (Filtered)       91
Mild Hypoxia (Suspicious)        57
No Hypoxia (Normal)              42
Severe Hypoxia (Pathological)    30
Name: count, dtype: int64


In [14]:
UNINT = "Uninterpretable (Filtered)"
MAP_3 = {
    "No Hypoxia (Normal)": 0,
    "Mild Hypoxia (Suspicious)": 1,
    "Severe Hypoxia (Pathological)": 2
}

# 1) Build X
X = np.array([
    np.stack([row["FHR"], row["UC"]], axis=0)
    for _, row in merged_df.iterrows()
], dtype=np.float32)[..., np.newaxis]   # (N,2,1800,1)

# 2) Build y_interp (1=interpretable, 0=uninterpretable)
y_interp = (merged_df["Clinical_Label"] != UNINT).astype(int).values.astype(np.int32)

# 3) Build y_sev (0/1/2 for interpretable, -1 for uninterpretable)
def map_sev(lbl):
    if lbl == UNINT:
        return -1
    return MAP_3[lbl]

y_sev = merged_df["Clinical_Label"].apply(map_sev).values.astype(np.int32)

print("X:", X.shape)
print("Severity counts:", dict(zip(*np.unique(y_sev, return_counts=True))))
print("Interp counts:", dict(zip(*np.unique(y_interp, return_counts=True))))


X: (220, 2, 1800, 1)
Severity counts: {-1: 91, 0: 42, 1: 57, 2: 30}
Interp counts: {0: 91, 1: 129}


In [15]:
'''
Filter to interpretable cases and prepare multi-class severity labels
'''

# Mask: keep only interpretable samples (y_sev != -1)
mask_interpretable = (y_sev != -1)

X = X[mask_interpretable]
y_sev = y_sev[mask_interpretable]

print("After filtering uninterpretable:")
print("X:", X.shape)
print("Severity counts:", dict(zip(*np.unique(y_sev, return_counts=True))))

# Multi-class labels: 0 = Normal, 1 = Mild, 2 = Severe
y_mc = y_sev.astype(np.int32)

After filtering uninterpretable:
X: (129, 2, 1800, 1)
Severity counts: {0: 42, 1: 57, 2: 30}


# Build Model

In [16]:
# Define better ordinal decoding function
def decode_ordinal_better(y_ord_pred, thr_ge1=0.5, thr_ge2=0.5):
    """
    Decode ordinal predictions using threshold-based approach.

    Parameters
    ----------
    y_ord_pred : array-like of shape (N, 2)
        Sigmoid outputs for the two ordinal tasks:
        - y_ord_pred[:, 0] ≈ P(class >= 1)
        - y_ord_pred[:, 1] ≈ P(class >= 2)
    thr_ge1 : float, default=0.5
        Threshold for deciding if class >= 1.
    thr_ge2 : float, default=0.5
        Threshold for deciding if class >= 2.

    Returns
    -------
    np.ndarray of shape (N,)
        Predicted class labels 0, 1, or 2.
    """
    p_ge1 = y_ord_pred[:, 0]  # P(class >= 1)
    p_ge2 = y_ord_pred[:, 1]  # P(class >= 2)
    
    # Optionally enforce constraint: P(class >= 1) >= P(class >= 2)
    p_ge2 = np.minimum(p_ge2, p_ge1)
    
    # Threshold-based decoding:
    # - If P(class >= 1) < thr_ge1 -> class 0
    # - Else if P(class >= 2) < thr_ge2 -> class 1
    # - Else -> class 2
    class_preds = np.zeros_like(p_ge1, dtype=int)
    
    # Class 0 where we are not confident it's >= 1
    class_0_mask = p_ge1 < thr_ge1
    class_preds[class_0_mask] = 0
    
    # For the rest, check if it's >= 2
    remaining_mask = ~class_0_mask
    class_2_mask = remaining_mask & (p_ge2 >= thr_ge2)
    class_preds[class_2_mask] = 2
    
    # Remaining (>=1 but <2) are class 1
    class_1_mask = remaining_mask & ~class_2_mask
    class_preds[class_1_mask] = 1
    
    return class_preds

In [ ]:
# Tune ordinal thresholds per-fold to maximize QWK on a validation set
import numpy as np
from sklearn.metrics import cohen_kappa_score

def tune_ordinal_thresholds_qwk(y_ord_val_pred, y_val, 
                                     thr_ge1_grid=None, thr_ge2_grid=None,
                                     enforce_order=True, verbose=False):
    """
    Grid-search thresholds for decode_ordinal_better to maximize QWK.

    Parameters
    ----------
    y_ord_val_pred : array-like of shape (N, 2)
        Sigmoid outputs for the two ordinal tasks on the *validation* set.
    y_val : array-like of shape (N,)
        True ordinal labels in {0, 1, 2} for the validation set.
    thr_ge1_grid : iterable of float, optional
        Candidate thresholds for thr_ge1 (P(class >= 1)).
        If None, defaults to np.linspace(0.2, 0.8, 13).
    thr_ge2_grid : iterable of float, optional
        Candidate thresholds for thr_ge2 (P(class >= 2)).
        If None, defaults to np.linspace(0.2, 0.8, 13).
    enforce_order : bool, default=True
        If True, only consider threshold pairs with thr_ge2 <= thr_ge1.
    verbose : bool, default=False
        If True, print the best thresholds and QWK.

    Returns
    -------
    best_thr_ge1 : float
        Best threshold for P(class >= 1).
    best_thr_ge2 : float
        Best threshold for P(class >= 2).
    best_qwk : float
        Best quadratic weighted kappa obtained on the validation set.
    """
    y_val = np.asarray(y_val)
    if thr_ge1_grid is None:
        thr_ge1_grid = np.linspace(0.2, 0.8, 13)
    if thr_ge2_grid is None:
        thr_ge2_grid = np.linspace(0.2, 0.8, 13)

    best_qwk = -np.inf
    best_thr_ge1, best_thr_ge2 = 0.5, 0.5

    for t1 in thr_ge1_grid:
        for t2 in thr_ge2_grid:
            if enforce_order and (t2 > t1):
                continue
            # Decode with candidate thresholds
            y_pred_val = decode_ordinal_better(y_ord_val_pred, thr_ge1=t1, thr_ge2=t2)
            # Constraint: predictions should not collapse to fewer than 3 classes
            vals, cnts = np.unique(y_pred_val, return_counts=True)
            if len(vals) < 3:
                continue
            if cnts.min() < 2:  # optional: require at least 2 samples per class
                continue
            # Compute QWK for valid threshold pairs
            qwk = cohen_kappa_score(y_val, y_pred_val, weights="quadratic")
            if qwk > best_qwk:
                best_qwk = qwk
                best_thr_ge1, best_thr_ge2 = t1, t2

    if verbose:
        print(f"Best thresholds: thr_ge1={best_thr_ge1:.3f}, thr_ge2={best_thr_ge2:.3f}, QWK={best_qwk:.4f}")

    return best_thr_ge1, best_thr_ge2, best_qwk

In [18]:
from keras.layers import Concatenate

def build_ctg_ms_encoder(input_shape=(2, 1800, 1), dropout_rate=0.25):
    inputs = Input(shape=input_shape)

    x = Conv2D(4, (1,3), padding="same", use_bias=False)(inputs)
    x = BatchNormalization()(x)

    x = DepthwiseConv2D((2,1), depth_multiplier=2, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = AveragePooling2D((1,4))(x)
    x = Dropout(dropout_rate)(x)

    b1 = SeparableConv2D(8, (1,3), padding="same", use_bias=False)(x)
    b1 = BatchNormalization()(b1); b1 = Activation("relu")(b1)

    b2 = SeparableConv2D(8, (1,7), padding="same", use_bias=False)(x)
    b2 = BatchNormalization()(b2); b2 = Activation("relu")(b2)

    b3 = SeparableConv2D(8, (1,15), padding="same", use_bias=False)(x)
    b3 = BatchNormalization()(b3); b3 = Activation("relu")(b3)

    x = Concatenate(axis=-1)([b1,b2,b3])

    x = Conv2D(8, (1,1), use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = AveragePooling2D((1,4))(x)
    x = Dropout(dropout_rate)(x)

    feat = GlobalAveragePooling2D()(x)
    return inputs, feat


def build_multitask_model():
    """
    Single-head multi-class severity classifier: 3 classes (0,1,2).
    """
    inputs, feat = build_ctg_ms_encoder()
    severity_logits = Dense(3, activation="softmax", name="severity_softmax")(feat)
    return Model(inputs, severity_logits)

In [19]:
# (No longer needed: ordinal loss was for ordinal multi-head model)
def ordinal_bce_loss(y_true, y_pred):
    return tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred), axis=-1)

In [27]:
'''
10-fold CV for multi-class severity classification (interpretable only).
We now train a single-head 3-class classifier on labels y_mc ∈ {0,1,2}.
'''

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

acc_scores = []
macro_f1_scores = []
qwk_scores = []

In [ ]:
'''
Run 10-fold CV with the single-head multi-class severity model.
Thresholds for ordinal decoding are tuned per fold on a
validation split of the training data to maximize QWK.
'''

from sklearn.model_selection import train_test_split

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y_mc), 1):
    print(f"\n{'='*60}")
    print(f"Fold {fold}/{n_splits}")
    print(f"{'='*60}")

    # Split original training fold into train/validation for threshold tuning
    X_train_full, X_test = X[train_idx], X[test_idx]
    y_train_full, y_test = y_mc[train_idx], y_mc[test_idx]

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.2, stratify=y_train_full, random_state=42 + fold
    )

    # Class-balanced sample weights on training split
    class_counts = np.bincount(y_train, minlength=3)
    total = len(y_train)
    class_weights = {c: total / (3 * max(cnt, 1)) for c, cnt in enumerate(class_counts)}
    w_train = np.array([class_weights[c] for c in y_train], dtype=np.float32)

    # Build and train model
    K.clear_session()
    model = build_multitask_model()

    model.compile(
        optimizer=Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    early_stop = EarlyStopping(
    monitor="val_loss",
    patience=25,
    min_delta=1e-4,
    restore_best_weights=True,
    verbose=0
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=10,
        min_lr=1e-6,
        verbose=0
    )

    print("Training...", end=" ", flush=True)
    history = model.fit(
    X_train, y_train,
    sample_weight=w_train,
    validation_data=(X_val, y_val),
    batch_size=8,
    epochs=300,
    callbacks=[early_stop, reduce_lr],
    verbose=0
    )
    print(f"Done! ({len(history.history['loss'])} epochs)")

    # Tune thresholds on validation split to maximize QWK
    y_proba_val = model.predict(X_val, verbose=0)  # shape (N_val, 3)
    p_ge1_val = 1.0 - y_proba_val[:, 0]           # P(class >= 1)
    p_ge2_val = y_proba_val[:, 2]                 # P(class >= 2)
    y_ord_val_pred = np.stack([p_ge1_val, p_ge2_val], axis=1)

    best_thr_ge1, best_thr_ge2, best_qwk_val = tune_ordinal_thresholds_qwk(
        y_ord_val_pred, y_val, verbose=True
    )

    # Evaluate on test fold using tuned thresholds
    y_proba_test = model.predict(X_test, verbose=0)
    p_ge1_test = 1.0 - y_proba_test[:, 0]
    p_ge2_test = y_proba_test[:, 2]
    y_ord_test_pred = np.stack([p_ge1_test, p_ge2_test], axis=1)
    y_pred = decode_ordinal_better(
        y_ord_test_pred, thr_ge1=best_thr_ge1, thr_ge2=best_thr_ge2
    )

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    qwk = cohen_kappa_score(y_test, y_pred, weights="quadratic")

    acc_scores.append(acc)
    macro_f1_scores.append(macro_f1)
    qwk_scores.append(qwk)

    print(f"Acc={acc:.3f}, Macro-F1={macro_f1:.3f}, QWK={qwk:.3f}")

print("\n" + "="*60)
print("Cross-validation completed!")
print("="*60)


Fold 1/10
Training... Done! (188 epochs)
Best thresholds: thr_ge1=0.700, thr_ge2=0.200, QWK=0.6316
Acc=0.385, Macro-F1=0.333, QWK=0.283

Fold 2/10
Training... Done! (159 epochs)
Best thresholds: thr_ge1=0.500, thr_ge2=0.400, QWK=0.4783
Acc=0.308, Macro-F1=0.254, QWK=0.013

Fold 3/10
Training... Done! (143 epochs)
Best thresholds: thr_ge1=0.600, thr_ge2=0.400, QWK=0.4146
Acc=0.538, Macro-F1=0.514, QWK=0.494

Fold 4/10
Training... Done! (178 epochs)
Best thresholds: thr_ge1=0.700, thr_ge2=0.450, QWK=0.3654
Acc=0.615, Macro-F1=0.613, QWK=0.376

Fold 5/10
Training... Done! (190 epochs)
Best thresholds: thr_ge1=0.550, thr_ge2=0.550, QWK=0.4962
Acc=0.615, Macro-F1=0.618, QWK=0.624

Fold 6/10
Training... Done! (149 epochs)
Best thresholds: thr_ge1=0.550, thr_ge2=0.550, QWK=0.3419
Acc=0.462, Macro-F1=0.471, QWK=0.474

Fold 7/10
Training... Done! (194 epochs)
Best thresholds: thr_ge1=0.750, thr_ge2=0.500, QWK=0.6842
Acc=0.385, Macro-F1=0.370, QWK=0.559

Fold 8/10
Training... Done! (112 epochs)

In [29]:
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
    cohen_kappa_score, roc_auc_score,
)
import numpy as np

# Full-dataset evaluation of the final trained model (from last CV fold)
y_proba_full = model.predict(X, verbose=0)  # shape (N,3)
y_pred_full = np.argmax(y_proba_full, axis=1)

print("=== Severity multi-class (ALL interpretable samples) ===")
print("Acc:", accuracy_score(y_mc, y_pred_full))
print("Macro-F1:", f1_score(y_mc, y_pred_full, average="macro", zero_division=0))
print("QWK:", cohen_kappa_score(y_mc, y_pred_full, weights="quadratic"))

print("Confusion:\n", confusion_matrix(y_mc, y_pred_full))
print("Report:\n", classification_report(y_mc, y_pred_full, zero_division=0))

# One-vs-rest ROC-AUC for each class
for c in [0, 1, 2]:
    y_true_bin = (y_mc == c).astype(int)
    if len(np.unique(y_true_bin)) < 2:
        print(f"Class {c}: AUC N/A (only one class present)")
        continue
    auc_c = roc_auc_score(y_true_bin, y_proba_full[:, c])
    print(f"Class {c}: AUC={auc_c:.3f}")

=== Severity multi-class (ALL interpretable samples) ===
Acc: 0.5813953488372093
Macro-F1: 0.5779059267121877
QWK: 0.5302764666217128
Confusion:
 [[32  4  6]
 [17 20 20]
 [ 3  4 23]]
Report:
               precision    recall  f1-score   support

           0       0.62      0.76      0.68        42
           1       0.71      0.35      0.47        57
           2       0.47      0.77      0.58        30

    accuracy                           0.58       129
   macro avg       0.60      0.63      0.58       129
weighted avg       0.63      0.58      0.57       129

Class 0: AUC=0.841
Class 1: AUC=0.644
Class 2: AUC=0.840


In [30]:
# Aggregate CV metrics for multi-class severity
import numpy as np

print("\n" + "="*60)
print("OVERALL MULTI-CLASS SEVERITY CV RESULTS")
print("="*60)

print(f"Mean Acc     : {np.mean(acc_scores):.3f} ± {np.std(acc_scores):.3f}")
print(f"Mean Macro-F1: {np.mean(macro_f1_scores):.3f} ± {np.std(macro_f1_scores):.3f}")
print(f"Mean QWK     : {np.mean(qwk_scores):.3f} ± {np.std(qwk_scores):.3f}")


OVERALL MULTI-CLASS SEVERITY CV RESULTS
Mean Acc     : 0.481 ± 0.100
Mean Macro-F1: 0.464 ± 0.119
Mean QWK     : 0.423 ± 0.193
